# 03 · 비즈니스 임베딩: Cluster와 Domain Adapter

**목표**: 고객 문의를 임베딩하여 비슷한 업무를 cluster로 묶고, 라벨된 과거 사례로 도메인별 semantic routing 품질을 높입니다. 예시는 결제·물류·컴플라이언스·기술 지원 문의를 사용합니다.

실행 전 루트 `.env`의 `EMBED_MODEL`을 현재 AI Core에 배포된 embedding 모델명으로 설정하세요. 학습용 사례는 합성 데이터이며, 운영에서는 개인정보를 제거하고 사람이 확정한 업무 큐 라벨만 사용합니다. 이 실습은 cluster·추천·평가만 생성하며 외부 업무 큐나 상태를 변경하지 않습니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 새 문의 분석과 추천 큐 확인 (UI)</h2>
<p style="margin: 7px 0;">아래 UI에 새 업무 문의를 입력하고 분석 및 추천을 누르세요. 합성 사례를 기준으로 cluster, holdout 평가, 추천 큐와 라벨별 유사도를 JSON으로 확인합니다.</p>
<ol style="margin: 7px 0 0; padding-left: 1.3em;"><li>새 업무 문의를 입력합니다.</li><li>분석 결과에서 추천 큐와 라벨별 유사도를 확인합니다.</li><li>낮은 유사도·작은 margin·신규 업무 유형은 자동 배정 대신 사람 검토로 보냅니다.</li></ol>
<p style="margin: 7px 0;">UI는 embedding 분석과 추천만 수행하며, 외부 업무 큐나 상태를 변경하지 않습니다.</p>
  </div>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "1_gen_ai_hub" / "src").exists())
for module_path in (project_root / "1_gen_ai_hub" / "src", project_root / "2_gen_ai_hub_llm"):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

import importlib
import src.notebook_ui as notebook_ui
import src.business_embedding.lab_core as business_embedding_core

# 이전 실행의 로컬 UI를 닫고, 방금 저장한 코드를 다시 읽습니다.
if "business_embedding_ui" in globals():
    business_embedding_ui.close()
importlib.reload(business_embedding_core)
importlib.reload(notebook_ui)

from src.notebook_ui import build_business_embedding_ui
from src.business_embedding.lab_core import load_business_cases, run_business_embedding_lab

business_embedding_ui = build_business_embedding_ui(
    run_business_embedding_lab,
    cases=load_business_cases(),
    n_clusters=4,
)
business_embedding_ui.launch(
    inline=True,
    share=False,
    show_error=True,
    server_name="127.0.0.1",
    width="100%",
    height=920,
)

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">실행 전 확인</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li><code>EMBED_MODEL</code>이 현재 AI Core에 배포된 embedding 모델명이다.</li><li>예제 사례에는 개인정보나 실제 고객 데이터가 포함되지 않는다.</li><li>결과는 추천·평가용이며, 실제 업무 큐 배정이나 상태 변경을 실행하지 않는다.</li></ul>
  </div>
</div>

<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 테스트된 코어 API로 전체 흐름 보기</h2>
<p style="margin: 7px 0;">아래 셀은 fixture 로드, embedding 호출, cluster, holdout 분리, adapter 학습, 새 문의 추천을 노트북에서 다시 구현하지 않습니다. 테스트된 공개 API의 반환값만 실행·확인합니다.</p>
  </div>
</div>

In [5]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "1_gen_ai_hub" / "src").exists())
for module_path in (project_root / "1_gen_ai_hub" / "src", project_root / "2_gen_ai_hub_llm"):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

from lab_common.config import load_settings
from src.business_embedding.lab_core import load_business_cases, run_business_embedding_lab

settings = load_settings()
cases = load_business_cases()

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. Cluster·holdout 평가·새 문의 추천 실행</h2>
<p style="margin: 7px 0;">예제는 4개 cluster 가설을 사용합니다. cluster ID에는 업무 의미가 없으므로, 실제 운영에서는 사람이 cluster 텍스트를 검토한 뒤에만 큐 이름을 부여해야 합니다. holdout 정확도도 작은 합성 예제의 참고값일 뿐입니다.</p>
  </div>
</div>

In [6]:
new_ticket = "운송장에는 배송 완료로 나오지만 고객이 물건을 받지 못했습니다."
outcome = run_business_embedding_lab(
    ticket=new_ticket,
    cases=cases,
    settings=settings,
    n_clusters=4,
)
outcome

{'model': 'amazon--titan-embed-image',
 'embedding_dimensions': 1024,
 'case_count': 12,
 'clusters': [{'id': 'PAY-001',
   'text': '고객 카드 결제가 두 번 승인되어 중복 청구 환불을 요청합니다.',
   'label': 'payment',
   'cluster': 0},
  {'id': 'PAY-002',
   'text': '세금계산서 금액과 결제 금액이 달라 정산 확인이 필요합니다.',
   'label': 'payment',
   'cluster': 3},
  {'id': 'PAY-003',
   'text': '자동이체 출금이 실패해 미납금 납부 방법을 문의합니다.',
   'label': 'payment',
   'cluster': 3},
  {'id': 'LOG-001',
   'text': '주문한 상품의 배송이 예정일보다 이틀 지연되고 있습니다.',
   'label': 'logistics',
   'cluster': 3},
  {'id': 'LOG-002',
   'text': '배송된 상자에 주문 상품 일부가 누락되어 재출고를 요청합니다.',
   'label': 'logistics',
   'cluster': 3},
  {'id': 'LOG-003',
   'text': '운송장에는 배달 완료지만 고객은 상품을 받지 못했다고 합니다.',
   'label': 'logistics',
   'cluster': 3},
  {'id': 'CMP-001',
   'text': '법인 고객의 사업자등록증이 만료되어 KYC 서류 갱신이 필요합니다.',
   'label': 'compliance',
   'cluster': 2},
  {'id': 'CMP-002',
   'text': '고객 신분증 정보와 계정 정보가 불일치하여 본인 확인을 요청합니다.',
   'label': 'compliance',
   'cluster': 3},
  {'id':

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. 반환된 근거 읽기</h2>
<p style="margin: 7px 0;">반환값에서 cluster inertia, holdout 평가, 새 문의의 추천 큐와 라벨별 유사도를 확인합니다. 최고 유사도와 1·2위 차이가 낮은 사례는 운영 워크플로에서 사람 검토 또는 일반 문의 큐로 보내야 합니다.</p>
  </div>
</div>

In [7]:
print(f"embedding model: {outcome['model']}")
print(f"case count: {outcome['case_count']} | dimensions: {outcome['embedding_dimensions']}")
print(f"cluster inertia: {outcome['cluster_inertia']:.4f}")
print(f"holdout accuracy: {outcome['holdout']['accuracy']:.3f}")
print("추천 큐:", outcome['recommendation']['recommended_queue'])
print("라벨별 유사도:", outcome['recommendation']['label_similarity'])
print(outcome['action'])

embedding model: amazon--titan-embed-image
case count: 12 | dimensions: 1024
cluster inertia: 0.0816
holdout accuracy: 0.500
추천 큐: logistics
라벨별 유사도: {'logistics': 0.9231110274407632, 'payment': 0.916305321380954, 'compliance': 0.8815012927438466, 'technical': 0.8409167163161317}
추천과 평가만 생성했습니다. 외부 업무 큐나 상태는 변경하지 않았습니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">운영에 연결할 때</h2>
<ol style="margin: 7px 0 0; padding-left: 1.3em;"><li>합성 사례를 개인정보를 제거한 과거 문의와 사람이 확정한 큐 라벨로 교체합니다.</li><li>시간 기준 holdout과 충분한 라벨 데이터로 cluster 수·adapter 성능을 평가합니다.</li><li>낮은 유사도·작은 margin·신규 업무 유형은 자동 배정 대신 사람 검토로 보냅니다.</li><li>실제 업무 큐 변경은 이 추천 모듈 밖에서 권한 검증과 감사 로그를 거쳐 수행합니다.</li></ol>
  </div>
</div>